# accel-sim silicon anchor — recomputation profiler

Times a real 4-layer chain (GPT-2's own GEMM shapes: 768→768→768→3072→768) forward+backward,
**stored** (normal autograd) vs **recompute** (`torch.utils.checkpoint` — PyTorch's real
implementation of activation checkpointing).

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

Writes `recompute_profile.json`, prints it, and auto-downloads it. Bring that file back and run:

```bash
python validate/silicon/compare_recompute.py recompute_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
# NOTE: on a Colab T4 (Turing) or V100 use DTYPE = "fp16" -- those GPUs have
# NO bf16 tensor cores and bf16 silently falls back to a ~20x-slower kernel.
DTYPE   = "fp16"          # "bf16" | "fp16" | "fp32"
TOKENS  = 8 * 1024        # batch 8 x seq 1024
ITERS   = 50
WARMUP  = 15
OUT     = "recompute_profile.json"

# GPT-2's own shapes, chained sequentially (mirrors q_proj -> attn_out ->
# mlp_up -> mlp_down; matches simulator/workloads.py's dimensions).
CHAIN_DIMS = [(768, 768), (768, 768), (768, 3072), (3072, 768)]


In [ ]:
_DTYPES = {"bf16": "bfloat16", "fp16": "float16", "fp32": "float32"}

import torch
import torch.nn as nn
import torch.utils.checkpoint as checkpoint
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
dtype = getattr(torch, _DTYPES[DTYPE])
M = TOKENS
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}   dtype={DTYPE}   tokens={M}   chain={CHAIN_DIMS}   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
import statistics

def bench(use_checkpoint):
    layers = [nn.Linear(k, n, bias=False).to(device=device, dtype=dtype)
              for k, n in CHAIN_DIMS]
    params = [p for l in layers for p in l.parameters()]
    x = torch.randn(M, CHAIN_DIMS[0][0], device=device, dtype=dtype, requires_grad=True)

    def forward_stored():
        out = x
        for l in layers:
            out = l(out)
        return out

    def forward_checkpointed():
        out = x
        for l in layers:
            out = checkpoint.checkpoint(l, out, use_reentrant=False)
        return out

    forward_fn = forward_checkpointed if use_checkpoint else forward_stored

    fwd, bwd = [], []
    for i in range(WARMUP + ITERS):
        for p in params:
            p.grad = None
        x.grad = None
        # 4 events: bracket the loss reduction into "forward" so it isn't
        # silently counted as part of "backward".
        ev = [torch.cuda.Event(enable_timing=True) for _ in range(4)]
        ev[0].record()
        out = forward_fn()
        ev[1].record()
        loss = out.float().square().mean()
        ev[2].record()
        loss.backward()
        ev[3].record()
        torch.cuda.synchronize()
        if i >= WARMUP:
            fwd.append(ev[0].elapsed_time(ev[2]))
            bwd.append(ev[2].elapsed_time(ev[3]))

    def stat(v):
        return {"mean_ms": statistics.fmean(v),
                "std_ms": statistics.pstdev(v) if len(v) > 1 else 0.0}
    return {"forward": stat(fwd), "backward": stat(bwd)}


In [ ]:
stored = bench(use_checkpoint=False)
print(f"  stored       fwd {stored['forward']['mean_ms']:8.3f}  bwd {stored['backward']['mean_ms']:8.3f} ms")
recompute = bench(use_checkpoint=True)
print(f"  recompute    fwd {recompute['forward']['mean_ms']:8.3f}  bwd {recompute['backward']['mean_ms']:8.3f} ms")
d_total = ((recompute["forward"]["mean_ms"] + recompute["backward"]["mean_ms"]) -
           (stored["forward"]["mean_ms"] + stored["backward"]["mean_ms"]))
stored_total = stored["forward"]["mean_ms"] + stored["backward"]["mean_ms"]
print(f"  measured delta: {d_total:+.3f} ms ({d_total / stored_total * 100:+.1f}%)")


In [ ]:
import json, platform

out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "dtype": _DTYPES[DTYPE], "tokens": M, "chain_dims": CHAIN_DIMS,
    "iters": ITERS, "warmup": WARMUP,
    "stored": stored, "recompute": recompute, "host": platform.platform(),
}
with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print(f"\n===== {OUT} (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
